# Classificação de Riscos em Notícias para Instituições Financeiras
---
**Autor:** Alexsander Lima Cerqueira Avidago  
**Orientador:** Prof. Ítalo João Bolqui Dutra  
**Curso:** MBA USP/Esalq – Data Science and Analytics

Este notebook implementa um pipeline de Machine Learning para classificar automaticamente notícias externas em categorias de risco relevantes para instituições financeiras. São comparados três algoritmos: **Decision Tree**, **Random Forest** e **Naive Bayes**.

---

## 1. Configuração do Ambiente e Importações

In [ ]:
# Configuração da sessão Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master('local[*]') \
    .appName("classificacao_riscos_v2") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

In [ ]:
# Importações gerais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuração visual global
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'font.family': 'sans-serif',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
})

# Paleta profissional
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4', '#8B8BAE']
COLORS_METRICS = ['#264653', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51']

sns.set_theme(style="whitegrid", palette=COLORS)
print("✅ Ambiente configurado com sucesso.")

## 2. Carregamento e Exploração dos Dados

In [ ]:
# Leitura do dataset
df_pandas = pd.read_excel('noticias_tcc_2025.xlsx', engine='openpyxl').fillna("Nihil")
df_pandas = df_pandas.replace(r'\n', ' ', regex=True)

# Limpeza de caracteres de controle
for col in df_pandas.columns:
    if col in ["noticia", "summary"]:
        df_pandas[col] = df_pandas[col].str.replace(r'\\r|\\n|\\r\\n|_x000D_', '', regex=True)

# Salvar como CSV intermediário
csv = bytes(df_pandas.to_csv(index=False), encoding='utf-8')
with open('noticias_tcc_2025.csv', 'wb') as f:
    f.write(csv)

print(f"Dataset carregado: {df_pandas.shape[0]} registros, {df_pandas.shape[1]} colunas")
print(f"Colunas: {list(df_pandas.columns)}")
df_pandas.head()

In [ ]:
# Carregamento no Spark DataFrame
df = spark.read.option("delimiter", ",").option("quote", '"').option("escape", '"') \
    .csv("noticias_tcc_2025.csv", header=True, inferSchema=True)

# Limpeza de espaços
from pyspark.sql import functions as F
for col in df.columns:
    df = df.withColumn(col, F.trim(F.col(col)))

print(f"Spark DataFrame: {df.count()} registros")
df.printSchema()

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
# Distribuição das categorias de risco
dist = df.groupBy('classificacao').count().orderBy('classificacao').toPandas()
dist.columns = ['Classificação', 'Quantidade']

print("\n📊 Distribuição das categorias de risco:")
print(dist.to_string(index=False))
print(f"\nTotal: {dist['Quantidade'].sum()} notícias")

In [ ]:
# GRÁFICO 1: Distribuição das categorias de risco
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Categorias e mapeamento
categorias_nomes = {
    'c1': 'C1 - Risco\nEconômico',
    'c2': 'C2 - Risco\nPolítico', 
    'c3': 'C3 - Risco\nFinanceiro',
    'c4': 'C4 - Risco\nRegulatório',
    'c5': 'C5 - Risco\nCibernético',
    'c6': 'C6 - Risco\nSocioambiental',
    'c7': 'C7 - Sem\nRelação'
}

labels = [categorias_nomes.get(c, c) for c in dist['Classificação']]
valores = dist['Quantidade'].values

# Gráfico de barras
bars = axes[0].bar(labels, valores, color=COLORS[:len(labels)], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição das Categorias de Risco', fontweight='bold', pad=15)
axes[0].set_ylabel('Quantidade de Notícias')

# Adicionar valores nas barras
for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=10)

# Gráfico de pizza
percentuais = valores / valores.sum() * 100
wedges, texts, autotexts = axes[1].pie(
    valores, labels=None, autopct='%1.1f%%',
    colors=COLORS[:len(labels)], startangle=90,
    pctdistance=0.85, wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2)
)
axes[1].set_title('Proporção por Categoria', fontweight='bold', pad=15)

# Legenda
labels_legenda = [categorias_nomes.get(c, c).replace('\n', ' ') for c in dist['Classificação']]
axes[1].legend(labels_legenda, loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)

plt.tight_layout()
plt.savefig('grafico_distribuicao_categorias.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("📊 Gráfico salvo: grafico_distribuicao_categorias.png")

## 4. Pré-processamento de Texto

In [ ]:
import pyspark.sql.functions as f

# Selecionar colunas relevantes e filtrar registros vazios
df = df.select('id', 'noticia', 'classificacao')
df = df.filter(df['noticia'] != "Nihil")

# Remoção de caracteres especiais
df_2 = df.withColumn("txt_sem_especiais", f.regexp_replace("noticia", "[\"!%&'()*+-./:;<=>?@^_`´{|}~\\\\]", ""))
df_2 = df_2.withColumn("texto_limpo", f.trim(df_2.txt_sem_especiais))

print(f"Registros após limpeza: {df_2.count()}")
df_2.select('noticia', 'texto_limpo').show(3, truncate=80)

In [ ]:
# Pipeline de pré-processamento textual
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer
from pyspark.sql.types import IntegerType

countTokens = f.udf(lambda tokens: len(tokens), IntegerType())

# Definição das etapas
stop = StopWordsRemover.loadDefaultStopWords("portuguese")
tokenizer = Tokenizer(inputCol="texto_limpo", outputCol="tokens")
stopwords = StopWordsRemover(inputCol="tokens", outputCol="texto_final", stopWords=stop)
cv = CountVectorizer(inputCol="texto_final", outputCol="CountVec")
tfidf = IDF(inputCol="CountVec", outputCol="features")

pipeline_preproc = Pipeline(stages=[tokenizer, stopwords, cv, tfidf])
dados_transformados = pipeline_preproc.fit(df_2).transform(df_2)

# Contagem de tokens antes e depois do stopwords
dados_transformados.select("tokens", "texto_final") \
    .withColumn("Tokens_Original", countTokens(f.col("tokens"))) \
    .withColumn("Tokens_Limpos", countTokens(f.col("texto_final"))) \
    .select("Tokens_Original", "Tokens_Limpos") \
    .describe().show()

print("✅ Pipeline de pré-processamento aplicado com sucesso.")

## 5. Visualização: Nuvem de Palavras

In [ ]:
# Nuvem de palavras com visual profissional
from wordcloud import WordCloud

amostra = dados_transformados.select('texto_final').sample(fraction=0.5, seed=101)
tudo = [noticia['texto_final'] for noticia in amostra.collect()]
texto_junto = ' '.join([' '.join(palavras) for palavras in tudo])

fig, ax = plt.subplots(figsize=(16, 8))

wordcloud = WordCloud(
    background_color='#1a1a2e',
    colormap='cool',
    width=1600,
    height=800,
    max_words=150,
    collocations=False,
    prefer_horizontal=0.8,
    min_font_size=8,
    max_font_size=120,
    contour_width=2,
    contour_color='#16213e'
).generate(texto_junto)

ax.imshow(wordcloud, interpolation='bilinear')
ax.axis("off")
ax.set_title('Nuvem de Palavras — Corpus de Notícias', fontweight='bold', fontsize=16, color='#333', pad=15)

plt.tight_layout()
plt.savefig('nuvem_palavras.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("📊 Gráfico salvo: nuvem_palavras.png")

## 6. Codificação da Variável Resposta e Divisão dos Dados

In [ ]:
from pyspark.ml.feature import StringIndexer

# Codificar a variável resposta
stringindexer = StringIndexer(inputCol="classificacao", outputCol="label")
dados = stringindexer.fit(df_2).transform(df_2)

print("Mapeamento de classes:")
dados.groupBy(['classificacao', 'label']).count().orderBy('label').show(truncate=False)

## 7. Treinamento dos Modelos de Classificação

Serão implementados três modelos de classificação supervisionada:
1. **Decision Tree** — Interpretável, baseado em ramificações lógicas
2. **Random Forest** — Ensemble de árvores, maior robustez
3. **Naive Bayes** — Probabilístico, eficaz para classificação de texto

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Dados base
dados = df
dados = dados.filter(dados['noticia'] != "Nihil")
dados = dados.withColumn("texto_transformado", f.regexp_replace("NOTICIA", "[\"!%&'()*+-./:;<=>?@^_`´{|}~\\\\]", ""))
dados = dados.withColumn("texto_limpo", f.trim(dados.texto_transformado))

# Definição das etapas do pipeline
labelIndexer = StringIndexer(inputCol="classificacao", outputCol="label").fit(dados)
tokenizer = Tokenizer(inputCol="texto_limpo", outputCol="tokens")
stopwords = StopWordsRemover(inputCol="tokens", outputCol="texto_final")
cv = CountVectorizer(inputCol="texto_final", outputCol="CountVec")
tfidf = IDF(inputCol="CountVec", outputCol="features")

# Classificadores
dt = DecisionTreeClassifier(featuresCol='features', labelCol='label', maxDepth=5)
rf = RandomForestClassifier(featuresCol='features', labelCol='label')
nb = NaiveBayes(featuresCol='features', labelCol='label')

# Divisão treino/teste (70/30)
treino_dados, teste_dados = dados.randomSplit([0.7, 0.3], seed=42)
print(f"Treino: {treino_dados.count()} registros | Teste: {teste_dados.count()} registros")

# Pipelines individuais
pipeline = Pipeline(stages=[labelIndexer, tokenizer, stopwords, cv, tfidf])
pipeline_dt = Pipeline(stages=[labelIndexer, tokenizer, stopwords, cv, tfidf, dt])
pipeline_rf = Pipeline(stages=[labelIndexer, tokenizer, stopwords, cv, tfidf, rf])
pipeline_nb = Pipeline(stages=[labelIndexer, tokenizer, stopwords, cv, tfidf, nb])

### 7.1 Árvore de Decisão (Decision Tree)

In [ ]:
# Treinamento e avaliação — Decision Tree
treino_dt = pipeline_dt.fit(treino_dados)
previsoes_dt = treino_dt.transform(teste_dados)

# Avaliadores
evaluator_acuracia = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

acuracia_dt = evaluator_acuracia.evaluate(previsoes_dt)
f1_dt = evaluator_f1.evaluate(previsoes_dt)
precision_dt = evaluator_precision.evaluate(previsoes_dt)
recall_dt = evaluator_recall.evaluate(previsoes_dt)

metricas_dt = pd.DataFrame({
    "Métrica": ["Acurácia", "F1-Score", "Precisão", "Recall"],
    "Valor": [acuracia_dt, f1_dt, precision_dt, recall_dt]
})

print("📊 Métricas — Decision Tree:")
print(metricas_dt.to_string(index=False))

### 7.2 Random Forest

In [ ]:
# Treinamento e avaliação — Random Forest
treino_rf = pipeline_rf.fit(treino_dados)
previsoes_rf = treino_rf.transform(teste_dados)

acuracia_rf = evaluator_acuracia.evaluate(previsoes_rf)
f1_rf = evaluator_f1.evaluate(previsoes_rf)
precision_rf = evaluator_precision.evaluate(previsoes_rf)
recall_rf = evaluator_recall.evaluate(previsoes_rf)

metricas_rf = pd.DataFrame({
    "Métrica": ["Acurácia", "F1-Score", "Precisão", "Recall"],
    "Valor": [acuracia_rf, f1_rf, precision_rf, recall_rf]
})

print("📊 Métricas — Random Forest:")
print(metricas_rf.to_string(index=False))

### 7.3 Naive Bayes

In [ ]:
# Treinamento e avaliação — Naive Bayes
treino_nb = pipeline_nb.fit(treino_dados)
previsoes_nb = treino_nb.transform(teste_dados)

acuracia_nb = evaluator_acuracia.evaluate(previsoes_nb)
f1_nb = evaluator_f1.evaluate(previsoes_nb)
precision_nb = evaluator_precision.evaluate(previsoes_nb)
recall_nb = evaluator_recall.evaluate(previsoes_nb)

metricas_nb = pd.DataFrame({
    "Métrica": ["Acurácia", "F1-Score", "Precisão", "Recall"],
    "Valor": [acuracia_nb, f1_nb, precision_nb, recall_nb]
})

print("📊 Métricas — Naive Bayes:")
print(metricas_nb.to_string(index=False))

## 8. Comparação dos Modelos e Visualização dos Resultados

In [ ]:
# Tabela comparativa consolidada
def avaliar_modelo(nome_modelo, previsoes):
    return {
        "Modelo": nome_modelo,
        "Acurácia": evaluator_acuracia.evaluate(previsoes),
        "F1-Score": evaluator_f1.evaluate(previsoes),
        "Precisão": evaluator_precision.evaluate(previsoes),
        "Recall": evaluator_recall.evaluate(previsoes)
    }

# Treinar novamente RF e NB para garantir consistência na comparação
treino_rf = pipeline_rf.fit(treino_dados)
previsoes_rf = treino_rf.transform(teste_dados)

treino_nb = pipeline_nb.fit(treino_dados)
previsoes_nb = treino_nb.transform(teste_dados)

resultados = [
    avaliar_modelo("Decision Tree", previsoes_dt),
    avaliar_modelo("Random Forest", previsoes_rf),
    avaliar_modelo("Naive Bayes", previsoes_nb)
]

metricas_df = pd.DataFrame(resultados)
print("\n" + "="*70)
print("         TABELA COMPARATIVA — DESEMPENHO DOS MODELOS")
print("="*70)
print(metricas_df.to_string(index=False))
print("="*70)

In [ ]:
# GRÁFICO 2: Comparativo de métricas dos modelos (Grouped Bar Chart)
fig, ax = plt.subplots(figsize=(14, 7))

metricas_cols = ['Acurácia', 'F1-Score', 'Precisão', 'Recall']
x = np.arange(len(metricas_cols))
width = 0.22

model_colors = ['#264653', '#2A9D8F', '#E76F51']
modelos = metricas_df['Modelo'].tolist()

for i, modelo in enumerate(modelos):
    valores = metricas_df[metricas_df['Modelo'] == modelo][metricas_cols].values.flatten()
    bars = ax.bar(x + i * width, valores, width, label=modelo, color=model_colors[i],
                  edgecolor='white', linewidth=1.5, zorder=3)
    # Valores nas barras
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Métrica', fontweight='bold')
ax.set_ylabel('Valor', fontweight='bold')
ax.set_title('Comparativo de Desempenho dos Modelos de Classificação', fontweight='bold', fontsize=14, pad=15)
ax.set_xticks(x + width)
ax.set_xticklabels(metricas_cols, fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.axhline(y=0.8, color='#E9C46A', linestyle='--', alpha=0.5, label='Meta 80%')

plt.tight_layout()
plt.savefig('grafico_comparativo_modelos.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("📊 Gráfico salvo: grafico_comparativo_modelos.png")

In [ ]:
# GRÁFICO 3: Gráfico Radar — Comparação dos Modelos
from matplotlib.patches import FancyBboxPatch

categories = ['Acurácia', 'F1-Score', 'Precisão', 'Recall']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

model_colors_radar = ['#264653', '#2A9D8F', '#E76F51']
model_fills = ['#26465330', '#2A9D8F30', '#E76F5130']

for i, modelo in enumerate(modelos):
    values = metricas_df[metricas_df['Modelo'] == modelo][categories].values.flatten().tolist()
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2.5, label=modelo, color=model_colors_radar[i], markersize=7)
    ax.fill(angles, values, alpha=0.15, color=model_colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9, color='gray')
ax.set_title('Perfil de Desempenho por Modelo', fontweight='bold', fontsize=14, pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grafico_radar_modelos.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("📊 Gráfico salvo: grafico_radar_modelos.png")

### 8.1 Matrizes de Confusão

In [ ]:
# GRÁFICO 4: Matrizes de confusão para cada modelo
from sklearn.metrics import confusion_matrix

def extrair_labels_predictions(previsoes_spark):
    """Extrai labels e predictions do DataFrame Spark para numpy arrays."""
    resultado = previsoes_spark.select("label", "prediction").toPandas()
    return resultado['label'].values, resultado['prediction'].values

# Classes ordenadas
classes = sorted(labelIndexer.labels)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
model_data = [
    ("Decision Tree", previsoes_dt),
    ("Random Forest", previsoes_rf),
    ("Naive Bayes", previsoes_nb)
]

for ax, (nome, previsoes) in zip(axes, model_data):
    y_true, y_pred = extrair_labels_predictions(previsoes)
    n_classes = len(np.unique(np.concatenate([y_true, y_pred])))
    cm = confusion_matrix(y_true, y_pred)
    
    # Normalizar por linha (recall por classe)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm)
    
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='YlOrRd', ax=ax,
                xticklabels=classes[:n_classes], yticklabels=classes[:n_classes],
                cbar_kws={'label': 'Recall', 'shrink': 0.8},
                linewidths=0.5, linecolor='white',
                annot_kws={"size": 9, "fontweight": "bold"})
    ax.set_title(nome, fontweight='bold', fontsize=13, pad=10)
    ax.set_xlabel('Previsto', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)
    ax.tick_params(axis='both', labelsize=8)

plt.suptitle('Matrizes de Confusão — Comparação dos Modelos', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('matrizes_confusao.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("📊 Gráfico salvo: matrizes_confusao.png")

In [ ]:
# GRÁFICO 5: Ranking de acurácia dos modelos
fig, ax = plt.subplots(figsize=(10, 4))

modelos_sorted = metricas_df.sort_values('Acurácia', ascending=True)
bars = ax.barh(modelos_sorted['Modelo'], modelos_sorted['Acurácia'], 
               color=model_colors[::-1], edgecolor='white', linewidth=1.5, height=0.5)

for bar, val in zip(bars, modelos_sorted['Acurácia']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2., 
            f'{val:.2%}', va='center', fontweight='bold', fontsize=12)

ax.set_xlim(0, 1)
ax.set_xlabel('Acurácia', fontweight='bold')
ax.set_title('Ranking de Acurácia dos Modelos', fontweight='bold', fontsize=14, pad=15)
ax.axvline(x=0.8, color='#E9C46A', linestyle='--', alpha=0.7, label='Meta 80%')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('grafico_ranking_acuracia.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("📊 Gráfico salvo: grafico_ranking_acuracia.png")

## 9. Validação Cruzada (Cross Validation)

Aplicação de Cross Validation com o CrossValidator do PySpark para validar o melhor modelo (Decision Tree) com tuning de hiperparâmetros.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Pipeline de CV com Decision Tree
tokenizer_cv = Tokenizer(inputCol="noticia", outputCol="tokens_noticia")
stopwords_cv = StopWordsRemover(inputCol="tokens_noticia", outputCol="texto_final_noticia")
cv_vec = CountVectorizer(inputCol="texto_final_noticia", outputCol="CountVec_noticia")
tfidf_cv = IDF(inputCol="CountVec_noticia", outputCol="features_noticia")
assembler = VectorAssembler(inputCols=["features_noticia"], outputCol="features")
labelIdx_cv = StringIndexer(inputCol="classificacao", outputCol="label", handleInvalid="keep")
dt_cv = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)

pipeline_cv = Pipeline(stages=[
    labelIdx_cv, tokenizer_cv, stopwords_cv, cv_vec, tfidf_cv, assembler, dt_cv
])

# Grade de parâmetros
paramGrid = ParamGridBuilder() \
    .addGrid(dt_cv.maxDepth, [5, 10]) \
    .build()

evaluator_cv = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Cross Validator com 5 folds
crossval = CrossValidator(
    estimator=pipeline_cv,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_cv,
    numFolds=5
)

treino_cv, teste_cv = df_2.randomSplit([0.7, 0.3], seed=42)
print("⏳ Executando Cross Validation (5 folds × 2 parâmetros = 10 treinamentos)...")
cvModel = crossval.fit(treino_cv)

previsoes_cv = cvModel.transform(teste_cv)
acuracia_cv = evaluator_cv.evaluate(previsoes_cv)
print(f"\n✅ Acurácia com Cross Validation: {acuracia_cv:.4f}")

bestMaxDepth = cvModel.bestModel.stages[-1].getOrDefault("maxDepth")
print(f"🏆 Melhor maxDepth selecionado: {bestMaxDepth}")

## 10. Aplicação: Classificação de Novas Notícias

Utilização do modelo Naive Bayes (melhor desempenho) para classificar novas notícias sem classificação prévia.

In [ ]:
# Carregar nova base de notícias
try:
    nova_base = pd.read_excel('noticias_novas.xlsx', engine='openpyxl').fillna("Nihil")
    nova_base = nova_base.replace(r'\n', ' ', regex=True)

    csv = bytes(nova_base.to_csv(index=False), encoding='utf-8')
    with open('nova_base.csv', 'wb') as f:
        f.write(csv)
    
    nova_base_spark = spark.read.csv("nova_base.csv", header=True, sep=',')
    nova_base_spark = nova_base_spark.withColumn(
        "texto_transformado",
        f.regexp_replace("noticia", "[\"!%&'()*+-./:;<=>?@^_`´{|}~\\\\]", "")
    )
    nova_base_spark = nova_base_spark.withColumn("texto_limpo", f.trim(nova_base_spark.texto_transformado))
    
    # Aplicar modelo Naive Bayes treinado
    modelo_nb = pipeline_nb.fit(treino_dados)
    previsoes_novas = modelo_nb.transform(nova_base_spark)
    
    # Converter labels numéricos de volta para nomes de classes
    from pyspark.ml.feature import IndexToString
    converter = IndexToString(inputCol="prediction", outputCol="classe_prevista", labels=labelIndexer.labels)
    resultado_final = converter.transform(previsoes_novas)
    
    print("\n📋 Classificação das Novas Notícias (Modelo: Naive Bayes):")
    resultado_final.select("id", "titulo", "classe_prevista").show(truncate=False)
    
except FileNotFoundError:
    print("⚠️ Arquivo 'noticias_novas.xlsx' não encontrado. Pule esta célula ou adicione o arquivo.")

## 11. Conclusão

| Modelo | Acurácia | F1-Score | Precisão | Recall |
|--------|----------|----------|----------|--------|
| Decision Tree | 0.6593 | 0.6311 | 0.6801 | 0.6593 |
| Random Forest | 0.5735 | 0.4629 | 0.5319 | 0.5735 |
| **Naive Bayes** | **0.8064** | **0.8078** | **0.8182** | **0.8064** |

**O modelo Naive Bayes apresentou o melhor desempenho geral**, com acurácia de 80,64% e F1-Score de 0,8078, sendo selecionado como modelo final para classificação automática de novas notícias.

---
*Notebook desenvolvido como parte do Trabalho de Conclusão de Curso (TCC) — MBA USP/Esalq*